# 05 — Dataset statistics: figure source data

**Project:** RAOD++ · **Phase:** F5 · **Protocol:** v1.0

Produces the CSVs behind the dataset-characterisation figures (the v1 manuscript's
Fig. 2 and Fig. 3 are replaced by leaner, readable versions drawn from these files
in F9). Nothing is plotted for the manuscript here — only raw numbers are stored,
so any figure can be redrawn later without touching the GPU.

Outputs to `output/figdata/`:
- `dataset_class_freq.csv` — instances per class per split
- `dataset_boxes.csv` — every box: centre, size, area, COCO size band
- `dataset_image_stats.csv` — per-image object counts and density

**Runtime:** 2–4 min. CPU only — safe to run while COCO downloads.


## [Cell 3] Config and paths

In [ ]:
# ===== Cell 3 =====
from pathlib import Path
import yaml, numpy as np, pandas as pd

ROOT = Path.home() / "RAODpp"
OUT = ROOT / "output"; FIGDATA = OUT / "figdata"; FIGDATA.mkdir(parents=True, exist_ok=True)
DATA_YAML = ROOT / "dataset" / "visdrone_raodpp.yaml"

dcfg = yaml.safe_load(open(DATA_YAML))
BASE = Path(dcfg["path"]); NAMES = dcfg["names"]
SPLITS = {"train": dcfg["train"], "val": dcfg["val"]}

print("dataset:", BASE)
print("classes:", len(NAMES))
for s, rel in SPLITS.items():
    print(f"  {s}: {len(list((BASE/rel).glob('*.jpg')))} images")


## [Cell 5] Read every label file into a flat table
One row per bounding box. 380k rows for VisDrone — small enough to keep as CSV and
large enough to answer any distributional question a reviewer might raise.

In [ ]:
# ===== Cell 5 =====
rows = []
for split, rel in SPLITS.items():
    lbl_dir = Path(str(BASE/rel).replace("/images", "/labels"))
    assert lbl_dir.exists(), f"labels not found at {lbl_dir}"
    for lp in sorted(lbl_dir.glob("*.txt")):
        for line in open(lp):
            p = line.split()
            if len(p) != 5:
                continue
            c, cx, cy, w, h = int(p[0]), *map(float, p[1:])
            rows.append((split, lp.stem, c, NAMES[c], cx, cy, w, h))

boxes = pd.DataFrame(rows, columns=["split","image","class_id","class_name",
                                    "cx","cy","w","h"])
boxes["area_norm"] = boxes.w * boxes.h
print(boxes.groupby("split").size())
boxes.head()


## [Cell 7] COCO size bands
Small/medium/large are defined on absolute pixel area. VisDrone images vary in size,
so we approximate with the modal resolution and record which resolution was assumed —
the assumption is stated in the manuscript rather than hidden.

In [ ]:
# ===== Cell 7 =====
from PIL import Image
from collections import Counter

sizes = Counter()
for split, rel in SPLITS.items():
    for p in list((BASE/rel).glob("*.jpg"))[:400]:      # sample is enough for the mode
        sizes[Image.open(p).size] += 1
MODAL_W, MODAL_H = sizes.most_common(1)[0][0]
print("modal resolution:", MODAL_W, "x", MODAL_H, "| distinct sizes sampled:", len(sizes))

boxes["area_px"] = boxes.area_norm * MODAL_W * MODAL_H
boxes["size_band"] = pd.cut(boxes.area_px, [-1, 32**2, 96**2, np.inf],
                            labels=["small","medium","large"])
boxes["assumed_res"] = f"{MODAL_W}x{MODAL_H}"
print(boxes.groupby(["split","size_band"], observed=True).size().unstack(fill_value=0))


## [Cell 9] Write the three figure-source CSVs

In [ ]:
# ===== Cell 9 =====
boxes.to_csv(FIGDATA/"dataset_boxes.csv", index=False)

freq = (boxes.groupby(["split","class_id","class_name"]).size()
        .rename("instances").reset_index())
freq["share"] = freq.groupby("split")["instances"].transform(lambda s: s/s.sum())
freq.to_csv(FIGDATA/"dataset_class_freq.csv", index=False)

img_stats = (boxes.groupby(["split","image"])
             .agg(n_objects=("class_id","size"),
                  mean_area_px=("area_px","mean"),
                  min_area_px=("area_px","min"),
                  n_small=("size_band", lambda s: (s=="small").sum()),
                  n_classes=("class_id","nunique")).reset_index())
img_stats.to_csv(FIGDATA/"dataset_image_stats.csv", index=False)

for f in ["dataset_boxes.csv","dataset_class_freq.csv","dataset_image_stats.csv"]:
    p = FIGDATA/f
    print(f"{f:28s} {p.stat().st_size/1e6:7.1f} MB")
print()
print(freq[freq.split=="train"].sort_values("instances", ascending=False)
      [["class_name","instances","share"]].to_string(index=False))


## [Cell 11] Headline numbers for the Experimental Setup section
These printed values go into the manuscript text and the data card. They are read
from the CSVs above, never retyped from memory.

In [ ]:
# ===== Cell 11 =====
tr = boxes[boxes.split=="train"]; va = boxes[boxes.split=="val"]
stats = {
    "train_boxes": len(tr), "val_boxes": len(va),
    "train_images_with_labels": tr.image.nunique(),
    "objects_per_image_train": round(len(tr)/tr.image.nunique(), 1),
    "small_share_train": round((tr.size_band=="small").mean(), 4),
    "median_area_px_train": round(tr.area_px.median(), 1),
    "class_imbalance_ratio": round(tr.class_name.value_counts().max() /
                                   tr.class_name.value_counts().min(), 1),
    "assumed_resolution": f"{MODAL_W}x{MODAL_H}",
}
pd.Series(stats).to_csv(FIGDATA/"dataset_headline_stats.csv", header=["value"])
for k, v in stats.items():
    print(f"{k:28s} {v}")


**Next:** once E0 finishes, `06_lowconf_mining.ipynb` uses the per-detection dumps
written by `figdata.capture_predictions` to set the class-adaptive threshold τ and mine
the low-confidence crop set.